Add __pow__ to the Value class so you can compute x ** n. Verify that d/dx(x^3) at x=2 equals 12.0.

In [3]:
class Value:
    def __init__(self,data,children=(),op=''):
        self.data=data
        self.grad=0.0
        self._backward=lambda:None #private/internal — don't touch unless you know what you're doing."
        self._prev=set(children)
        self._op=op
        
    def __repr__(self):
        return f"Value(data={self.data:.4f},grad={self.grad:.4f})"
    
    def __pow__(self, n):
            out = Value(self.data ** n, (self,), f'**{n}')
            def _backward():
                self.grad += n * (self.data ** (n - 1)) * out.grad
            out._backward = _backward
            return out
        
    def backward(self):
        # Topological order all children in the graph
        topo = []
        visited = set()
        def build_topo(v):
            if v not in visited:
                visited.add(v)
                for child in v._prev:
                    build_topo(child)
                topo.append(v)
        build_topo(self)
        
        # Go one variable at a time and apply chain rule
        self.grad = 1.0
        for v in reversed(topo):
            v._backward()

In [4]:
x=Value(2.0)
y=x**3
y.backward()

print(f"x = {x}")
print(f"y = x^3 = {y}")
print(f"dy/dx at x=2: {x.grad}")
print(f"Expected (3*x^2 = 3*4): 12.0")
print(f"Verification: {'PASS' if abs(x.grad - 12.0) < 1e-6 else 'FAIL'}")


x = Value(data=2.0000,grad=12.0000)
y = x^3 = Value(data=8.0000,grad=1.0000)
dy/dx at x=2: 12.0
Expected (3*x^2 = 3*4): 12.0
Verification: PASS
